# 最佳融合模型完整指标（便于与 AcrPred 全面对比）

**目的**：加载 `roc_curve_demo.ipynb` 训练并保存的 **ProteinBERT_PSSM1110** 融合模型预测结果与完整评估指标，便于与 AcrPred (IJBM 2023) 做全面对比。

**与 AcrPred 论文的对应关系**：
- 数据来源与划分与 AcrPred 一致（anti-CRISPRdb + 统一资源，CD-HIT 去冗余，同一 train/test 划分）。
- AcrPred 原文仅报告 **AUC, ACC, SN (Sensitivity), SP (Specificity)**；本 notebook 额外展示 **AUPRC, F1, MCC, Brier, ECE** 及验证集最优阈值，便于多维度对比与撰写论文。

## 1. 依赖与加载预测结果

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

COMPARISON_RESULTS_DIR = '/home/nemophila/projects/protein_bert/Comparison/results'

data = np.load(os.path.join(COMPARISON_RESULTS_DIR, 'ours_predictions.npz'))
y_test = data['y_true']
test_prob = data['y_prob']

with open(os.path.join(COMPARISON_RESULTS_DIR, 'ours_fusion_metrics.json')) as f:
    saved = json.load(f)
metrics = saved['metrics']
thr = metrics['Threshold']

print(f'Loaded {len(y_test)} test samples, threshold={thr:.4f}')

## 2. 完整指标与混淆矩阵

In [ ]:
for k, v in metrics.items():
    print(f'{k}: {v}')

y_pred = (test_prob >= thr).astype(int)
cm = confusion_matrix(y_test, y_pred)
print('\nConfusion matrix (test):')
print(cm)

## 3. 与 AcrPred (IJBM 2023) 对比表

AcrPred 原文在**同一数据构建**的独立测试集上报告：AUC=0.952, SN=0.923, SP=0.877, ACC=0.881；未报告 AUPRC/F1/MCC/Brier/ECE。下表便于论文中直接引用。

In [ ]:
acrpred_paper = {
    'AUC': 0.952,
    'ACC': 0.881,
    'SN': 0.923,
    'SP': 0.877,
    'AUPRC': None,
    'F1': None,
    'MCC': None,
    'Brier': None,
    'ECE': None,
}

display_keys = ['AUC', 'ACC', 'SN', 'SP', 'AUPRC', 'F1', 'MCC', 'Brier', 'ECE']
ours = {k: round(metrics[k], 4) for k in display_keys}

comparison = pd.DataFrame({
    'AcrPred (IJBM 2023)': [acrpred_paper[k] if acrpred_paper[k] is not None else '—' for k in display_keys],
    'Ours (ProteinBERT+PSSM1110)': [ours[k] for k in display_keys],
}, index=display_keys)
comparison